# 07 · Comparar regiones y olas con gráficos interactivos

Comparamos dos tipos de segmentos:

- **Regiones:** las cuatro del US Census Bureau (Noreste, Medio Oeste, Sur y Oeste).
- **Periodos:** las tres olas que cortamos en el notebook 03 (primavera de 2020, verano de 2020 y
  otoño-invierno de 2020-21).

Los gráficos clave son **interactivos** (`plotly`): al pasar el ratón se ven las cifras exactas, se puede hacer
zoom y hay botones para cambiar de variable o de ola. Todas las cifras son **por cada 100.000 habitantes**
(notebook 05), porque las regiones tienen tamaños muy distintos: el Sur tiene casi el doble de habitantes que el
Noreste.

Cada gráfico se guarda también como página HTML en `reports/interactivos/`, que se abre en cualquier navegador.

## 0. Preparación

In [1]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

from src.graficos import AZUL, NARANJA, TEXTO, TEXTO_SUAVE, VERDE, formato_miles
from src.interactivos import COLORES_REGION, ESCALA_MUERTES, guardar_html, registrar_plantilla
from src.poblacion import por_100k
from src.variables import OLAS, REGIONES

registrar_plantilla()
pio.renderers.default = "plotly_mimetype+notebook_connected"

RUTA_FINAL = RAIZ / "data" / "processed" / "covid_estados_final.csv"
CARPETA_HTML = RAIZ / "reports" / "interactivos"
if not RUTA_FINAL.exists():
    raise FileNotFoundError("Ejecuta antes python -m src.pipeline o los notebooks 01, 02, 03, 05 y 06: generan "
        + str(RUTA_FINAL.relative_to(RAIZ)))

datos = pd.read_csv(RUTA_FINAL, parse_dates=["fecha", "semana"])
estados = datos[~datos["es_territorio"]].copy()
estados["ola"] = pd.Categorical(estados["ola"], categories=list(OLAS), ordered=True)
ORDEN_REGIONES = list(REGIONES)
CORTES = pd.to_datetime(list(OLAS.values())[:2])
estados.shape

(18995, 36)

## 1. Cada estado en su región

La región se asignó en el notebook 03 (`asignar_region()` en `src/variables.py`) con la división oficial del US
Census Bureau. DC cuenta como estado del Sur. Comprobamos que los 51 tienen región y cuánta población suma cada
una.

In [2]:
por_estado = estados.drop_duplicates("estado")[["estado", "nombre_estado", "region", "poblacion"]]
assert por_estado["region"].isin(ORDEN_REGIONES).all() and len(por_estado) == 51

poblacion_region = por_estado.groupby("region")["poblacion"].sum().reindex(ORDEN_REGIONES)
resumen_regiones = pd.DataFrame({
    "estados": por_estado["region"].value_counts().reindex(ORDEN_REGIONES),
    "habitantes": poblacion_region,
    "% de la población": poblacion_region / poblacion_region.sum() * 100,
})
(resumen_regiones.style.format(precision=0, thousands=".", subset=["habitantes"])
 .format("{:.1f} %", decimal=",", subset=["% de la población"]))

,estados,habitantes,% de la población
region,,,
Noreste,9,57.430.477,"17,3 %"
Medio Oeste,12,68.969.794,"20,8 %"
Sur,17,126.465.281,"38,1 %"
Oeste,13,78.661.381,"23,7 %"


In [3]:
mapa_regiones = px.choropleth(
    por_estado.assign(habitantes=por_estado["poblacion"].map(formato_miles)),
    locations="estado", locationmode="USA-states", scope="usa",
    color="region", color_discrete_map=COLORES_REGION, category_orders={"region": ORDEN_REGIONES},
    hover_name="nombre_estado", hover_data={"estado": False, "region": True, "habitantes": True},
    labels={"region": "Región", "habitantes": "Habitantes"},
)
mapa_regiones.update_traces(marker_line_color="white", marker_line_width=1)
mapa_regiones.update_layout(
    title="Las cuatro regiones del US Census Bureau",
    legend=dict(title="", orientation="h", y=1.02, x=0.01, yanchor="bottom"),
    geo=dict(bgcolor="rgba(0,0,0,0)", lakecolor="rgba(0,0,0,0)"),
    height=460, margin=dict(l=10, r=10, t=90, b=10),
)
guardar_html(mapa_regiones, CARPETA_HTML / "1_mapa_regiones.html", "Regiones del US Census Bureau")
mapa_regiones.show()

El **Sur** es la región más grande (17 estados y el 38 % de la población) y el **Noreste** la más pequeña
(9 estados y el 17 %). Por eso comparar cifras absolutas entre regiones no tiene sentido: el Sur casi siempre
"ganaría". A partir de aquí, todo va por cada 100.000 habitantes.

## 2. Las regiones en el tiempo

Para cada región y día sumamos los casos, hospitalizados y muertes de sus estados, calculamos la media de 7 días
(notebook 06) y la dividimos entre la población de la región.

En las muertes dejamos fuera los **días de atraso** marcados en el notebook 02 (volcados de certificados
antiguos). Esas muertes son reales, pero no ocurrieron el día en que se publicaron: el volcado de Ohio de febrero
de 2021 movería el pico del Medio Oeste dos meses. Los totales de los apartados siguientes sí las incluyen.

In [4]:
estados["muertes_sin_atrasos"] = estados["muertes_nuevas"].where(~estados["atraso_muertes"])

medidas = {
    "casos_nuevos": ("Casos", "Casos nuevos al día por cada 100.000 habitantes (media de 7 días)"),
    "hospitalizados_actuales": ("Hospitalizados", "Hospitalizados a la vez por cada 100.000 habitantes (media de 7 días)"),
    "muertes_sin_atrasos": ("Muertes", "Muertes al día por cada 100.000 habitantes (media de 7 días)"),
}

serie_region = estados.groupby(["region", "fecha"])[list(medidas)].sum(min_count=1).reset_index()
for columna in medidas:
    media_7d = serie_region.groupby("region")[columna].transform(lambda s: s.rolling(7, min_periods=4).mean())
    serie_region[columna + "_100k"] = por_100k(media_7d, serie_region["region"].map(poblacion_region))
serie_region = serie_region[serie_region["fecha"] >= "2020-03-15"]

picos = {}
for columna, (nombre, _) in medidas.items():
    fila = serie_region.loc[serie_region.groupby("region")[columna + "_100k"].idxmax()]
    picos[nombre] = fila.set_index("region")[["fecha", columna + "_100k"]].set_axis(["fecha del pico", "valor"], axis=1)
pd.concat(picos, axis=1).reindex(ORDEN_REGIONES).style.format(
    {(n, "fecha del pico"): lambda f: f.strftime("%d-%m-%Y") for n in picos} | {(n, "valor"): "{:.1f}" for n in picos},
    decimal=",")

In [5]:
def anotaciones_finales(columna):
    """Nombre de la región al final de cada línea, separados para que no se pisen."""
    ultimos = (serie_region.sort_values("fecha").groupby("region").tail(1).set_index("region")[columna + "_100k"]
               .sort_values())
    rango = serie_region[columna + "_100k"].max()
    posiciones, anterior = {}, None
    for region, valor in ultimos.items():
        y = valor if anterior is None else max(valor, anterior + rango * 0.06)
        posiciones[region], anterior = y, y
    return [dict(x=serie_region["fecha"].max(), y=y, text=f"<b>{region}</b>", xanchor="left", xshift=6,
                 showarrow=False, font=dict(color=COLORES_REGION[region], size=12)) for region, y in posiciones.items()]


def anotaciones_olas():
    nombres = [nombre.split(" · ")[1] for nombre in OLAS]
    inicios = [serie_region["fecha"].min(), *CORTES]
    return [dict(x=inicio, y=1.0, yref="paper", text=nombre, xanchor="left", xshift=4, showarrow=False,
                 font=dict(color=TEXTO_SUAVE, size=11)) for inicio, nombre in zip(inicios, nombres)]


tiempo = go.Figure()
for columna, (nombre, _) in medidas.items():
    for region in ORDEN_REGIONES:
        tramo = serie_region[serie_region["region"] == region]
        tiempo.add_trace(go.Scatter(
            x=tramo["fecha"], y=tramo[columna + "_100k"], name=region, legendgroup=region,
            line=dict(color=COLORES_REGION[region], width=2.5), visible=(columna == "casos_nuevos"),
            hovertemplate=f"{region}: %{{y:.1f}}<extra></extra>",
        ))

botones = []
for i, (columna, (nombre, titulo)) in enumerate(medidas.items()):
    visible = [j // len(ORDEN_REGIONES) == i for j in range(len(tiempo.data))]
    botones.append(dict(label=nombre, method="update",
                        args=[{"visible": visible}, {"title.text": titulo,
                                                     "annotations": anotaciones_olas() + anotaciones_finales(columna)}]))

for corte in CORTES:
    tiempo.add_vline(x=corte, line=dict(color=TEXTO_SUAVE, width=1, dash="dash"))
tiempo.update_layout(
    title=medidas["casos_nuevos"][1],
    annotations=anotaciones_olas() + anotaciones_finales("casos_nuevos"),
    title_y=0.97, title_yanchor="top",
    updatemenus=[dict(type="buttons", direction="right", buttons=botones, x=0.0, y=1.08, xanchor="left",
                      yanchor="bottom", showactive=True, bgcolor="white", bordercolor="#d0cfca")],
    hovermode="x unified", showlegend=False, height=540, margin=dict(r=110, t=130),
    xaxis=dict(rangeslider=dict(visible=True, thickness=0.06), tickformat="%m-%Y", hoverformat="%d-%m-%Y"),
)
guardar_html(tiempo, CARPETA_HTML / "2_regiones_en_el_tiempo.html", "Regiones en el tiempo")
tiempo.show()

Con los botones de arriba se cambia entre casos, hospitalizados y muertes; con la barra de abajo se elige un
periodo para hacer zoom.

- **Primavera de 2020: una ola casi solo del Noreste.** Sus muertes llegaron a 2,3 al día por cada 100.000
  habitantes (16-04-2020) y sus hospitalizados a 60, los dos valores más altos de cualquier región en todo el
  periodo, aunque en casos detectados nunca pasó de lo que vendría después. Las otras regiones apenas se movían.
- **Verano de 2020: el Sur y el Oeste.** El Noreste, ya de bajada, se queda plano todo el verano.
- **Otoño-invierno: todas a la vez, pero no al mismo ritmo.** El **Medio Oeste** subió antes que nadie: su pico de
  casos fue el 19-11-2020 (91 al día por cada 100.000, el más alto de todos) y el de muertes el 11-12-2020. El
  Sur, el Oeste y el Noreste tuvieron su pico de casos entre el 8 y el 13 de enero de 2021, casi dos meses
  después.
- **Hospitalizados:** se ve el mismo calendario, pero en la primavera la cifra se queda corta porque no todos los
  estados lo publicaban todavía (notebook 01).

## 3. Cada ola en el mapa

Ahora por estado: muertes por cada 100.000 habitantes en cada ola. Los tres mapas usan la **misma escala de
color** (de 0 a 200), así que se pueden comparar entre sí: un estado claro en verano y oscuro en invierno tuvo
de verdad muchas más muertes en invierno.

In [6]:
nombres_estado = por_estado.set_index("estado")["nombre_estado"]
region_estado = por_estado.set_index("estado")["region"]
poblacion_estado = por_estado.set_index("estado")["poblacion"]

estado_ola = estados.groupby(["estado", "ola"], observed=True)[["casos_nuevos", "muertes_nuevas"]].sum().reset_index()
estado_ola["muertes_100k"] = por_100k(estado_ola["muertes_nuevas"], estado_ola["estado"].map(poblacion_estado))
estado_ola["casos_100k"] = por_100k(estado_ola["casos_nuevos"], estado_ola["estado"].map(poblacion_estado))
estado_ola["nombre"] = estado_ola["estado"].map(nombres_estado)
estado_ola["region"] = estado_ola["estado"].map(region_estado)
estado_ola["puesto"] = estado_ola.groupby("ola", observed=True)["muertes_100k"].rank(ascending=False, method="min").astype(int)
print("Máximo de muertes por 100.000 en una ola:", round(estado_ola["muertes_100k"].max(), 1))

mapa_olas = go.Figure()
for i, ola in enumerate(OLAS):
    tramo = estado_ola[estado_ola["ola"] == ola]
    mapa_olas.add_trace(go.Choropleth(
        locations=tramo["estado"], locationmode="USA-states", z=tramo["muertes_100k"],
        zmin=0, zmax=200, colorscale=ESCALA_MUERTES, marker_line_color="white", marker_line_width=0.8,
        visible=(i == 0), name=ola,
        customdata=tramo[["nombre", "region", "casos_100k", "puesto"]],
        hovertemplate=("<b>%{customdata[0]}</b> (%{customdata[1]})<br>Muertes por 100.000: %{z:.0f}<br>"
                       "Casos por 100.000: %{customdata[2]:,.0f}<br>Puesto en la ola: %{customdata[3]} de 51"
                       "<extra></extra>"),
        colorbar=dict(title=dict(text="Muertes por<br>100.000 hab."), thickness=14, len=0.7),
    ))

botones = [dict(label=ola, method="update",
                args=[{"visible": [j == i for j in range(len(OLAS))]},
                      {"title.text": f"Muertes por cada 100.000 habitantes · ola {ola}"}])
           for i, ola in enumerate(OLAS)]
mapa_olas.update_layout(
    title=f"Muertes por cada 100.000 habitantes · ola {list(OLAS)[0]}",
    title_y=0.97, title_yanchor="top",
    updatemenus=[dict(type="buttons", direction="right", buttons=botones, x=0.01, y=1.04, xanchor="left",
                      yanchor="bottom", bgcolor="white", bordercolor="#d0cfca")],
    geo=dict(scope="usa", bgcolor="rgba(0,0,0,0)", lakecolor="rgba(0,0,0,0)"),
    height=500, margin=dict(l=10, r=10, t=110, b=10),
)
guardar_html(mapa_olas, CARPETA_HTML / "3_mapa_muertes_por_ola.html", "Muertes por ola en el mapa")
mapa_olas.show()

Máximo de muertes por 100.000 en una ola: 193.4


- **Primavera:** solo se oscurece la esquina noreste (Nueva Jersey, Nueva York, Connecticut, Massachusetts) y
  algún foco aislado como Luisiana o Michigan. El resto del país está casi en blanco.
- **Verano:** el mapa más claro de los tres. Los tonos medios se concentran en el Sur (Misisipi, Luisiana) y en
  Arizona.
- **Otoño-invierno:** casi todo el mapa se oscurece. Los más oscuros ya no están en la costa, sino en el centro
  del país: las Dakotas, con las cifras más altas de cualquier estado en cualquier ola.

## 4. Dentro de cada región: ¿todos los estados igual?

Una región es una media de estados que pueden ser muy distintos. En este gráfico cada punto es un estado; la caja
marca dónde está la mitad central de los estados de esa región.

In [7]:
dispersion = px.box(
    estado_ola, x="region", y="muertes_100k", color="region", facet_col="ola", points="all",
    category_orders={"region": ORDEN_REGIONES, "ola": list(OLAS)}, color_discrete_map=COLORES_REGION,
    hover_name="nombre", hover_data={"region": False, "ola": False, "muertes_100k": ":.0f"},
    labels={"muertes_100k": "Muertes por 100.000 hab.", "region": ""},
)
dispersion.for_each_annotation(lambda a: a.update(text=f"<b>{a.text.split('=')[-1]}</b>"))
dispersion.update_traces(marker=dict(size=7, line=dict(width=1, color="white")), jitter=0.4, pointpos=0,
                         boxmean=False, line_width=1.5)
dispersion.update_layout(
    title="Muertes por cada 100.000 habitantes de cada estado, por región y ola",
    showlegend=False, height=500, margin=dict(t=110),
)
dispersion.update_xaxes(tickangle=-30)
guardar_html(dispersion, CARPETA_HTML / "4_estados_por_region.html", "Estados por región y ola")
dispersion.show()

estado_ola.groupby(["ola", "region"], observed=True)["muertes_100k"].agg(["min", "median", "max"]).round(0)

min  median    max
ola                        region                          
1 · primavera 2020         Medio Oeste   8.0    17.0   59.0
                           Noreste       7.0    75.0  151.0
                           Oeste         1.0     5.0   27.0
                           Sur           5.0    13.0   74.0
2 · verano 2020            Medio Oeste   9.0    13.0   19.0
                           Noreste       0.0    11.0   26.0
                           Oeste         4.0    11.0   59.0
                           Sur          10.0    24.0   62.0
3 · otoño-invierno 2020-21 Medio Oeste  81.0   113.0  193.0
                           Noreste      23.0    81.0  135.0
                           Oeste        24.0    80.0  153.0
                           Sur          62.0    98.0  155.0

- **El Noreste de la primavera es la región más desigual:** de 7 muertes por 100.000 en el estado menos afectado
  a 151 en Nueva Jersey. Maine y Vermont, del mismo Noreste, no llegaron a 10, y New Hampshire se quedó en 21;
  la media de la región la hacen el área de Nueva York y el sur de Nueva Inglaterra.
- **En otoño-invierno, el Medio Oeste tiene el suelo más alto:** su estado menos afectado (81) está al nivel de
  la mediana del Noreste y del Oeste. La ola golpeó a toda la región, no solo a unos pocos estados.
- **El Oeste es muy variado:** Arizona fue de los peores estados del país en verano (2.º) y en otoño-invierno
  (4.º), mientras que Hawái estuvo entre los seis mejores en las tres olas.

Las medias por región son útiles para ver el calendario de la pandemia, pero esconden estas diferencias: para
decidir medidas concretas hay que bajar al estado.

## 5. Resumen por región y ola

Para cerrar, las cifras de cada segmento en una tabla. Además de casos y muertes por habitante, incluimos tres
indicadores que ayudan a interpretar las diferencias:

- **Tests por 100.000:** cuánto se buscaba el virus.
- **Positividad:** qué parte de los tests salía positiva. Si es alta, se estaban escapando casos.
- **Muertes por cada 100 casos:** la letalidad aparente (depende mucho de cuántos casos se detectan).

In [8]:
region_ola = estados.groupby(["region", "ola"], observed=True)[["casos_nuevos", "muertes_nuevas", "tests_nuevos"]].sum()
habitantes = region_ola.index.get_level_values("region").map(poblacion_region)
resumen = pd.DataFrame({
    "Casos / 100.000": por_100k(region_ola["casos_nuevos"], habitantes),
    "Muertes / 100.000": por_100k(region_ola["muertes_nuevas"], habitantes),
    "Tests / 100.000": por_100k(region_ola["tests_nuevos"], habitantes),
    "Positividad (%)": region_ola["casos_nuevos"] / region_ola["tests_nuevos"] * 100,
    "Muertes por 100 casos": region_ola["muertes_nuevas"] / region_ola["casos_nuevos"] * 100,
}).reindex(ORDEN_REGIONES, level="region")
(resumen.style
 .format(precision=0, thousands=".", subset=["Casos / 100.000", "Muertes / 100.000", "Tests / 100.000"])
 .format(precision=1, decimal=",", subset=["Positividad (%)", "Muertes por 100 casos"])
 .background_gradient(cmap="Oranges", subset=["Muertes / 100.000"]))

- **La primavera del Noreste fue también la de menos tests por caso:** 13,7 % de positividad y 7,2 muertes por
  cada 100 casos detectados. Muchos contagios no se llegaron a contar (notebook 03).
- **En el verano, el Noreste se controló y el Sur no:** el Noreste hacía 70 tests por cada caso (positividad del
  1,4 %); el Sur, 11 (9,4 %).
- **En otoño-invierno, el Noreste hizo el doble de tests por habitante que el Sur** (unos 124.000 frente a 61.000
  por cada 100.000) y tuvo la positividad más baja (5,0 % frente a 10,9 %). Con todo, las muertes por habitante
  quedaron cerca en las cuatro regiones (de 89 a 112).
- **La letalidad aparente se igualó:** en la ola de otoño-invierno, todas las regiones están en torno a 1,5
  muertes por cada 100 casos.

## Conclusiones

- **Cada ola tuvo su región:** el Noreste en la primavera de 2020, el Sur y el Oeste en el verano, y todas en
  otoño-invierno. En la primavera el Noreste llegó a las cifras diarias de muertes y hospitalizados por habitante
  más altas de todo el periodo.
- **El Medio Oeste fue el primero en la ola de otoño-invierno:** su pico de casos llegó casi dos meses antes que
  el de las demás regiones y fue el más alto por habitante.
- **Las regiones esconden mucha variedad:** en la primavera, Nueva Jersey tuvo 151 muertes por 100.000 y Maine o
  Vermont casi ninguna, todos en el Noreste.
- **Los tests cambian lo que se ve:** en invierno el Noreste hizo el doble de tests por habitante que el Sur y su
  positividad fue la mitad. Comparar casos entre regiones sin mirar cuánto se buscaba el virus puede engañar.
- Los cuatro gráficos interactivos están en `reports/interactivos/` y se abren en cualquier navegador.